# Phantom — Chest X-Ray Classifier
**EfficientNet-B4 · NIH ChestX-ray14 · 14 conditions**

### First-time setup
1. Runtime → Change runtime type → **A100 GPU + High RAM**
2. Run all cells — enter your Kaggle credentials when Cell 3 prompts you
3. Dataset downloads to **Google Drive** and stays there permanently

### Every session after that
Just run all cells. Dataset download is skipped automatically. Resumes from best checkpoint.

| Condition | Condition | Condition | Condition |
|---|---|---|---|
| No Finding | Atelectasis | Cardiomegaly | Effusion |
| Infiltration | Mass | Nodule | Pneumonia |
| Consolidation | Edema | Emphysema | Fibrosis |
| Pleural Thickening | Hernia | | |

In [ ]:
# ── Cell 1: Install + verify GPU ─────────────────────────
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'kaggle', 'scikit-learn', 'Pillow', 'tqdm'], check=True)

import torch
print(f'PyTorch  {torch.__version__}')
print(f'CUDA     {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      {torch.cuda.get_device_name(0)}')
    print(f'VRAM     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Cell 2: Mount Drive ───────────────────────────────────
# Dataset + checkpoints live here permanently — no re-downloading between sessions.
from google.colab import drive
import os, glob

drive.mount('/content/drive', force_remount=True)

DRIVE_DIR  = '/content/drive/MyDrive/Phantom'
DATA_DIR   = os.path.join(DRIVE_DIR, 'data')
IMAGES_DIR = os.path.join(DATA_DIR, 'images')

os.makedirs(IMAGES_DIR, exist_ok=True)

existing = glob.glob(os.path.join(IMAGES_DIR, '*.png'))
print(f'Drive root    : {DRIVE_DIR}')
print(f'Images on Drive: {len(existing):,}')

In [ ]:
# ── Cell 3: Download dataset (first time only) ────────────
# Images are saved to Google Drive — this cell is a no-op on all future sessions.
import shutil, json, getpass

existing = glob.glob(os.path.join(IMAGES_DIR, '*.png'))

if len(existing) < 10000:
    print('First-time download: NIH ChestX-ray14 → Google Drive')
    print('This takes ~30-40 min once, then never again.\n')

    # ── Enter Kaggle credentials ──────────────────────────
    kaggle_username = input('Kaggle username: ').strip()
    kaggle_key      = getpass.getpass('Kaggle API key:  ').strip()

    kaggle_dir = os.path.expanduser('~/.config/kaggle')
    os.makedirs(kaggle_dir, exist_ok=True)
    with open(os.path.join(kaggle_dir, 'kaggle.json'), 'w') as f:
        json.dump({'username': kaggle_username, 'key': kaggle_key}, f)
    os.chmod(os.path.join(kaggle_dir, 'kaggle.json'), 0o600)
    print('✓ Kaggle credentials set\n')

    # ── Download to local /content first (faster), then move to Drive ──
    tmp = '/content/nih_tmp'
    os.makedirs(tmp, exist_ok=True)
    result = subprocess.run(
        ['kaggle', 'datasets', 'download', 'nih-chest-xrays/data', '--unzip', '-p', tmp],
        capture_output=True, text=True
    )
    print(result.stdout[-2000:] if result.stdout else '')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-1000:])

    # ── Move images to Drive ──────────────────────────────
    print('\nMoving images to Google Drive (permanent storage)...')
    imgs = glob.glob(os.path.join(tmp, '**', '*.png'), recursive=True)
    for i, src in enumerate(imgs):
        dst = os.path.join(IMAGES_DIR, os.path.basename(src))
        if not os.path.exists(dst):
            shutil.move(src, dst)
        if (i + 1) % 5000 == 0:
            print(f'  {i+1:,}/{len(imgs):,}')

    # Move CSV
    for csv in glob.glob(os.path.join(tmp, '**', 'Data_Entry*.csv'), recursive=True):
        shutil.move(csv, os.path.join(DATA_DIR, os.path.basename(csv)))

    shutil.rmtree(tmp, ignore_errors=True)
    final = glob.glob(os.path.join(IMAGES_DIR, '*.png'))
    print(f'\nDone! {len(final):,} images saved to Drive permanently.')
else:
    print(f'✓ Dataset already on Drive ({len(existing):,} images) — skipping download.')

# Locate CSV
csv_candidates = glob.glob(os.path.join(DATA_DIR, 'Data_Entry*.csv'))
if not csv_candidates:
    csv_candidates = glob.glob(os.path.join(DATA_DIR, '**', 'Data_Entry*.csv'), recursive=True)
CSV_PATH = csv_candidates[0]
print(f'Labels CSV: {CSV_PATH}')

In [ ]:
# ── Cell 4: Config ────────────────────────────────────────
import numpy as np

CFG = dict(
    img_size        = 224,
    num_classes     = 14,
    dropout         = 0.3,
    epochs          = 30,
    freeze_epochs   = 3,      # warm up classifier before touching backbone
    batch_size      = 128,
    num_workers     = 4,      # conservative for Drive-backed reads
    lr_max          = 3e-4,
    weight_decay    = 1e-4,
    label_smoothing = 0.05,
    mixup_alpha     = 0.2,
    ema_decay       = 0.9998,
    grad_clip       = 1.0,
    save_every      = 3,
    model_name      = 'phantom_model_best.pth',
)

CONDITIONS = [
    'No Finding', 'Atelectasis', 'Cardiomegaly', 'Effusion',
    'Infiltration', 'Mass', 'Nodule', 'Pneumonia',
    'Consolidation', 'Edema', 'Emphysema', 'Fibrosis',
    'Pleural Thickening', 'Hernia'
]

BEST_PATH   = os.path.join(DRIVE_DIR, CFG['model_name'])
CKPT_PATH   = os.path.join(DRIVE_DIR, 'phantom_checkpoint.pth')

RESUME = os.path.exists(BEST_PATH)
if RESUME:
    print(f'✓ Checkpoint found ({os.path.getsize(BEST_PATH)/1e6:.1f} MB) — will resume')
else:
    print('No checkpoint — training from scratch')

In [ ]:
# ── Cell 5: Dataset ───────────────────────────────────────
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

all_images   = glob.glob(os.path.join(IMAGES_DIR, '*.png'))
IMAGE_LOOKUP = {os.path.basename(p): p for p in all_images}
print(f'Images available: {len(IMAGE_LOOKUP):,}')

df = pd.read_csv(CSV_PATH)
df = df[df['Image Index'].isin(IMAGE_LOOKUP)].copy()
for c in CONDITIONS:
    df[c] = df['Finding Labels'].apply(lambda x: 1 if c in x else 0)
print(f'Labelled rows: {len(df):,}')

train_df, val_df = train_test_split(
    df, test_size=0.15, random_state=42, stratify=df['No Finding']
)
print(f'Train: {len(train_df):,}  Val: {len(val_df):,}')

# Positive class weights
pos_counts  = train_df[CONDITIONS].sum()
neg_counts  = len(train_df) - pos_counts
pos_weights = torch.tensor(
    (neg_counts / pos_counts.clip(lower=1)).clip(upper=20).values.astype('float32')
)


class ChestXrayDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        img  = Image.open(IMAGE_LOOKUP[row['Image Index']]).convert('RGB')
        lbls = torch.FloatTensor(row[CONDITIONS].values.astype('float32'))
        return self.transform(img), lbls


sz = CFG['img_size']
train_tfm = T.Compose([
    T.Resize((sz + 32, sz + 32)),
    T.RandomCrop(sz),
    T.RandomHorizontalFlip(),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.3, contrast=0.3),
    T.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.1, scale=(0.02, 0.08)),
])
val_tfm = T.Compose([
    T.Resize((sz, sz)),
    T.ToTensor(),
    T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds = ChestXrayDataset(train_df, train_tfm)
val_ds   = ChestXrayDataset(val_df,   val_tfm)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=CFG['num_workers'], pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=CFG['batch_size'] * 2, shuffle=False,
                          num_workers=CFG['num_workers'], pin_memory=True)
print(f'Train batches: {len(train_loader)}  Val batches: {len(val_loader)}')

In [ ]:
# ── Cell 6: Model + optimiser ─────────────────────────────
import torch.nn as nn
import copy
from torchvision import models
from sklearn.metrics import roc_auc_score

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

def build_model():
    m    = models.efficientnet_b4(weights='IMAGENET1K_V1')
    in_f = m.classifier[1].in_features
    m.classifier = nn.Sequential(
        nn.Dropout(p=CFG['dropout']),
        nn.Linear(in_f, CFG['num_classes'])
    )
    return m

model = build_model().to(device)

# Resume
start_epoch = 0
best_auroc  = 0.0
if RESUME:
    state = torch.load(BEST_PATH, map_location=device, weights_only=True)
    if isinstance(state, dict) and 'model_state_dict' in state:
        model.load_state_dict(state['model_state_dict'], strict=False)
        start_epoch = state.get('epoch', 0)
        best_auroc  = state.get('auroc',  0.0)
    else:
        missing, _ = model.load_state_dict(state, strict=False)
        print(f'Weights loaded (missing keys: {len(missing)})')
        start_epoch = 10
        best_auroc  = 0.7250
    print(f'Resumed from epoch {start_epoch}, best AUROC {best_auroc:.4f}')

# EMA copy
ema_model = copy.deepcopy(model)
ema_model.eval()

def update_ema(ema, live):
    d = CFG['ema_decay']
    with torch.no_grad():
        for ep, lp in zip(ema.parameters(), live.parameters()):
            ep.data.mul_(d).add_(lp.data, alpha=1 - d)

# Loss with label smoothing
class SmoothedBCE(nn.Module):
    def __init__(self, pos_weight, s=0.05):
        super().__init__()
        self.bce = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
        self.s   = s
    def forward(self, logits, y):
        return self.bce(logits, y * (1 - self.s) + 0.5 * self.s)

criterion = SmoothedBCE(pos_weights.to(device), CFG['label_smoothing'])

# Phase 1: freeze backbone
for p in model.features.parameters():
    p.requires_grad = False

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3, weight_decay=CFG['weight_decay']
)

# Fixed amp — avoids NaN crash from deprecated torch.cuda.amp
scaler = torch.amp.GradScaler('cuda')

history = {'train_loss': [], 'val_auroc': []}
print(f'Params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M')
print(f'Training epochs {start_epoch+1} → {CFG["epochs"]}')
print(f'Frozen backbone for first {CFG["freeze_epochs"]} epochs, then full fine-tune')

In [ ]:
# ── Cell 7: Train ─────────────────────────────────────────
import json, time
from sklearn.metrics import f1_score

def mixup(x, y, alpha):
    if alpha <= 0: return x, y, y, 1.0
    lam = np.random.beta(alpha, alpha)
    idx = torch.randperm(x.size(0), device=x.device)
    return lam * x + (1-lam) * x[idx], y, y[idx], lam

@torch.no_grad()
def validate(mdl):
    mdl.eval()
    probs_all, lbls_all = [], []
    for imgs, lbls in val_loader:
        with torch.amp.autocast('cuda'):
            p = torch.sigmoid(mdl(imgs.to(device))).cpu().numpy()
        probs_all.append(p)
        lbls_all.append(lbls.numpy())
    P = np.vstack(probs_all)
    L = np.vstack(lbls_all)
    aurocs = [roc_auc_score(L[:,i], P[:,i])
              for i in range(len(CONDITIONS)) if L[:,i].sum() >= 5]
    return float(np.mean(aurocs)), P, L

def find_thresholds(P, L):
    out = {}
    for i, cond in enumerate(CONDITIONS):
        if L[:,i].sum() == 0:
            out[cond] = 0.5; continue
        best_t, best_f1 = 0.5, 0.0
        for t in np.arange(0.05, 0.80, 0.025):
            f1 = f1_score(L[:,i], (P[:,i] >= t).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_t = f1, float(t)
        out[cond] = round(best_t, 3)
    return out


scheduler = None
print('Starting training...\n')

for epoch in range(start_epoch, CFG['epochs']):
    # Unfreeze backbone after warm-up
    if epoch == CFG['freeze_epochs']:
        print(f'\nEpoch {epoch+1}: Unfreezing backbone, switching to OneCycleLR')
        for p in model.features.parameters():
            p.requires_grad = True
        remaining = CFG['epochs'] - epoch
        optimizer = torch.optim.AdamW(
            model.parameters(), lr=CFG['lr_max'] * 0.1,
            weight_decay=CFG['weight_decay']
        )
        scheduler = torch.optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=CFG['lr_max'],
            steps_per_epoch=len(train_loader), epochs=remaining,
            pct_start=0.15, div_factor=10, final_div_factor=100
        )

    # Train epoch
    model.train()
    total_loss = 0.0
    t0 = time.time()

    for step, (imgs, lbls) in enumerate(train_loader):
        imgs, lbls = imgs.to(device), lbls.to(device)
        imgs_m, ya, yb, lam = mixup(imgs, lbls, CFG['mixup_alpha'])

        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast('cuda'):
            logits = model(imgs_m)
            loss   = lam * criterion(logits, ya) + (1-lam) * criterion(logits, yb)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        scaler.step(optimizer)
        scaler.update()
        if scheduler: scheduler.step()
        update_ema(ema_model, model)
        total_loss += loss.item()

        if (step + 1) % 150 == 0:
            print(f'  [{epoch+1}/{CFG["epochs"]}] step {step+1}/{len(train_loader)}  loss={loss.item():.4f}')

    # Validate
    mean_auroc, val_probs, val_lbls = validate(ema_model)
    avg_loss = total_loss / len(train_loader)
    lr_now   = optimizer.param_groups[0]['lr']
    elapsed  = time.time() - t0
    history['train_loss'].append(avg_loss)
    history['val_auroc'].append(mean_auroc)

    marker = ''
    if mean_auroc > best_auroc:
        best_auroc = mean_auroc
        torch.save(ema_model.state_dict(), BEST_PATH)

        thresholds  = find_thresholds(val_probs, val_lbls)
        thresh_path = os.path.join(DRIVE_DIR, 'phantom_thresholds.json')
        with open(thresh_path, 'w') as f:
            json.dump(thresholds, f, indent=2)
        marker = ' ★  → model + thresholds saved'

    if (epoch + 1) % CFG['save_every'] == 0:
        torch.save({'epoch': epoch+1, 'auroc': best_auroc,
                    'model_state_dict': ema_model.state_dict()}, CKPT_PATH)

    phase = 'frozen' if epoch < CFG['freeze_epochs'] else 'full  '
    print(f'Ep {epoch+1:2d}/{CFG["epochs"]} [{phase}]  '
          f'loss={avg_loss:.4f}  AUROC={mean_auroc:.4f}  '
          f'lr={lr_now:.2e}  {elapsed:.0f}s{marker}')

print(f'\nDone. Best AUROC: {best_auroc:.4f}')

In [ ]:
# ── Cell 8: Final eval + results ─────────────────────────
import json

print('Loading best checkpoint...')
ema_model.load_state_dict(
    torch.load(BEST_PATH, map_location=device, weights_only=True)
)
mean_auroc, val_probs, val_lbls = validate(ema_model)

print(f'\nFinal mean AUROC: {mean_auroc:.4f}')
print(f'\n{"Condition":<22} {"AUROC":>7}')
print('─' * 32)
aurocs = []
for i, cond in enumerate(CONDITIONS):
    if val_lbls[:,i].sum() < 5:
        print(f'{cond:<22}   N/A')
        continue
    auc = roc_auc_score(val_lbls[:,i], val_probs[:,i])
    aurocs.append(auc)
    bar = '█' * int(auc * 25)
    print(f'{cond:<22} {auc:.3f}  {bar}')

# Print thresholds — paste these into model.py
thresh_path = os.path.join(DRIVE_DIR, 'phantom_thresholds.json')
if os.path.exists(thresh_path):
    with open(thresh_path) as f:
        thresholds = json.load(f)
    print('\n── Paste into backend/src/services/model.py ──')
    print('THRESHOLDS = {')
    for cond, t in thresholds.items():
        print(f"    '{cond}': {t},")
    print('}')

In [ ]:
# ── Cell 9: Download to laptop ────────────────────────────
# Drop phantom_model_best.pth into backend/models/ (rename to phantom_model.pth)
from google.colab import files

files.download(BEST_PATH)
if os.path.exists(thresh_path):
    files.download(thresh_path)

print('Downloaded:')
print(f'  {CFG["model_name"]}  →  rename to phantom_model.pth  →  place in backend/models/')
print('  phantom_thresholds.json  →  paste THRESHOLDS dict into backend/src/services/model.py')